# Tema 5 · Evaluación de predicción numérica

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

**Lechería Los Robles**, en Coihueco, entrega cada día su leche a una planta procesadora. La planta le pide comprometer con anticipación **cuántos litros** entregará cada día. El dueño probó un modelo que predice los litros con la temperatura, la lluvia, las vacas en ordeña y el concentrado, pero no sabe si confiar en él: «¿cuánto se equivoca?, ¿y qué pasa con los días en que se corta la luz?».

> **❓ La pregunta del cliente:** ¿Cuánto se equivoca, en litros, un modelo que predice la producción diaria, y cuál conviene usar?

## 🗂️ Los datos: `t5_lecheria.csv` (730 días de producción, con el clima, las vacas en ordeña, el concentrado y los litros)

| Columna | Qué significa |
|---|---|
| `fecha` | día (año-mes-día), desde el 01-07-2024 |
| `temperatura_max` | temperatura máxima del día, en °C |
| `lluvia_mm` | lluvia del día, en milímetros |
| `vacas_ordena` | vacas ordeñadas ese día |
| `concentrado_kg` | kilos de concentrado (alimento) por vaca |
| `litros` | **litros de leche entregados a la planta** ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t5_lecheria.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 730 filas y 6 columnas.

## 1. Conocer los datos

📘 Cada fila es **un día** de la lechería, desde julio de 2024 hasta junio de 2026: la temperatura máxima, la
lluvia, cuántas vacas se ordeñaron, cuánto concentrado (alimento) recibió cada vaca y los **litros** de leche que
se entregaron a la planta. Los litros son lo que se quiere predecir: es un **número**, no una clase.

In [ ]:
datos.head()

In [ ]:
print("Días registrados:", len(datos))
print(f"Litros por día: promedio {datos['litros'].mean():.0f}, mínimo {datos['litros'].min()}, máximo {datos['litros'].max()}")
# ✅ Debe mostrar: Días registrados: 730  /  Litros por día: promedio 2710, mínimo 807, máximo 3418

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(pd.to_datetime(datos["fecha"]), datos["litros"], lw=0.8)
plt.ylabel("Litros del día"); plt.title("Litros entregados a la planta, día a día")
plt.show()

### ✍️ Describan el gráfico: ¿la producción es pareja durante el año? ¿Ven días raros? ¿Qué podría explicarlos en una lechería?

_Su respuesta:_

## 2. Separar entrenamiento y prueba… respetando el tiempo

📘 En los temas anteriores se separaba al azar. Aquí los datos son una **serie de tiempo**: el modelo se va a usar
para predecir **días futuros**. Si se separa al azar, el modelo «ve» días de la semana siguiente y de la anterior a
cada día de prueba, y parece mejor de lo que es. Por eso se entrena con los **primeros 550 días** y se prueba con
los **últimos 180**, como si se estuviera en enero de 2026 mirando hacia adelante.

In [ ]:
X = datos[["temperatura_max", "lluvia_mm", "vacas_ordena", "concentrado_kg"]]
y = datos["litros"]
X_ent, X_pru = X[:550], X[550:]
y_ent, y_pru = y[:550], y[550:]
print("Entrenamiento:", datos["fecha"].iloc[0], "a", datos["fecha"].iloc[549], "·", len(y_ent), "días")
print("Prueba:", datos["fecha"].iloc[550], "a", datos["fecha"].iloc[-1], "·", len(y_pru), "días")
# ✅ Debe mostrar: Entrenamiento: 2024-07-01 a 2026-01-01 · 550 días  /  Prueba: 2026-01-02 a 2026-06-30 · 180 días

## 3. Cómo se mide el error de un número

📘 En clasificación se cuenta cuántas veces se acierta. Con números casi nunca se acierta exacto: se mide **cuánto
se equivoca**. El **error** de un día es real − predicho. Tres medidas:

- **MAE** (error absoluto medio): el promedio de los errores **sin signo**. Se lee directo en litros: «en promedio
  me equivoco en tantos litros».
- **RMSE** (raíz del error cuadrático medio): se elevan los errores **al cuadrado**, se promedian y se saca la raíz.
  También queda en litros, pero **castiga mucho más los errores grandes**.
- **R²**: compara el modelo con predecir siempre el promedio. **1** = perfecto; **0** = igual que el promedio;
  **negativo** = peor que el promedio.

La función `medir()` calcula las tres con su fórmula, paso a paso.

In [ ]:
def medir(real, predicho):
    """MAE, RMSE y R² de una predicción numérica, calculados con su fórmula."""
    real = np.array(real, dtype=float)
    predicho = np.array(predicho, dtype=float)
    error = real - predicho                                   # error de cada día
    mae = np.mean(np.abs(error))                              # promedio del error sin signo
    rmse = np.sqrt(np.mean(error ** 2))                       # raíz del promedio de los cuadrados
    r2 = 1 - np.sum(error ** 2) / np.sum((real - real.mean()) ** 2)   # 1 − (error del modelo / error del promedio)
    return mae, rmse, r2

def mostrar(nombre, real, predicho):
    mae, rmse, r2 = medir(real, predicho)
    print(f"{nombre}: MAE {mae:.0f} · RMSE {rmse:.0f} · R² {r2:.2f}")

# Línea base: predecir para cada día de prueba el promedio de los días de entrenamiento
pred_promedio = np.full(len(y_pru), y_ent.mean())
print(f"Promedio de entrenamiento: {y_ent.mean():.0f} litros")
mostrar("Siempre el promedio", y_pru, pred_promedio)
# ✅ Debe mostrar: Promedio de entrenamiento: 2747 litros  /  Siempre el promedio: MAE 200 · RMSE 327 · R² -0.26

### ✍️ Interpreten el MAE de la línea base en litros, como se lo dirían al dueño. El R² de la línea base sale cerca de 0 o negativo: ¿por qué?

_Su respuesta:_

## 4. Dos modelos: regresión lineal y árbol de regresión

📘 La **regresión lineal** (Semana 9) suma las columnas multiplicadas por coeficientes. El **árbol de regresión**
funciona como el árbol de decisión (Semana 7), pero cada hoja predice **el promedio de litros** de los días que
caen en ella. `max_depth=4` limita el árbol a 4 preguntas seguidas.

In [ ]:
lineal = LinearRegression().fit(X_ent, y_ent)
pred_lineal = lineal.predict(X_pru)

arbol = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X_ent, y_ent)
pred_arbol = arbol.predict(X_pru)

mostrar("Siempre el promedio", y_pru, pred_promedio)
mostrar("Regresión lineal   ", y_pru, pred_lineal)
mostrar("Árbol (prof. 4)    ", y_pru, pred_arbol)
# ✅ Debe mostrar: Siempre el promedio: MAE 200 · RMSE 327 · R² -0.26  /  Regresión lineal   : MAE 97 · RMSE 229 · R² 0.38  /  Árbol (prof. 4)    : MAE 156 · RMSE 304 · R² -0.09

In [ ]:
plt.figure(figsize=(9, 3.5))
fechas_pru = pd.to_datetime(datos["fecha"].iloc[550:])
plt.plot(fechas_pru, y_pru, color="black", lw=1, label="Real")
plt.plot(fechas_pru, pred_lineal, color="green", lw=1, label="Regresión lineal")
plt.plot(fechas_pru, pred_arbol, color="orange", lw=1, label="Árbol (prof. 4)")
plt.ylabel("Litros"); plt.legend(); plt.title("Periodo de prueba: real y predicho")
plt.show()

### ✍️ ¿Qué modelo se equivoca menos? Compárenlo con la línea base usando el MAE en litros. ¿Qué dice su R²? Miren el gráfico: ¿cómo se ve la predicción del árbol comparada con la de la regresión?

_Su respuesta:_

## 5. ¿Y si el árbol fuera más profundo?

📘 Un árbol más profundo tiene más hojas, es decir, más escalones. ¿Alcanza para ganarle a la regresión?

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
PROFUNDIDAD = 4        # ✏️ pruebe con 1, 2, 4, 8 y None (None = sin límite). Al terminar, deje 4.

arbol_prueba = DecisionTreeRegressor(max_depth=PROFUNDIDAD, random_state=0).fit(X_ent, y_ent)
mostrar(f"Árbol de profundidad {PROFUNDIDAD}", y_pru, arbol_prueba.predict(X_pru))
mostrar("Regresión lineal (para comparar)", y_pru, pred_lineal)

**✏️ Anoten aquí sus resultados:**

| Profundidad | MAE (litros) | RMSE (litros) | R² | ¿Le gana a la lineal? |
|:-:|:-:|:-:|:-:|:-:|
| 1 | | | | |
| 2 | | | | |
| 4 | | | | |
| 8 | | | | |
| None | | | | |

### ✍️ Según su tabla, ¿con alguna profundidad el árbol le gana a la regresión lineal? ¿Qué pasa con profundidad 1 y sin límite? ¿Qué concluyen sobre «el modelo más complejo»?

_Su respuesta:_

## 6. Los días de falla: MAE contra RMSE

📘 La celda busca los 3 días de prueba en que la regresión se equivocó más, y luego mide el error **sin esos días**.

In [ ]:
error_abs = np.abs(y_pru.values - pred_lineal)
peores = np.argsort(-error_abs)[:3]
for i in sorted(peores):
    print(datos["fecha"].iloc[550 + i], "· real", y_pru.iloc[i], "· predicho", int(round(pred_lineal[i])))

normales = np.ones(len(y_pru), dtype=bool)
normales[peores] = False
print()
mostrar("Con todos los días ", y_pru, pred_lineal)
mostrar("Sin los 3 días raros", y_pru[normales], pred_lineal[normales])
# ✅ Debe mostrar: 2026-02-21 · real 1045 · predicho 2636  /  2026-04-02 · real 963 · predicho 2479  /  2026-06-01 · real 897 · predicho 2701  /    /  Con todos los días : MAE 97 · RMSE 229 · R² 0.38  /  Sin los 3 días raros: MAE 71 · RMSE 88 · R² 0.81

### ✍️ ¿Cuánto cambian el MAE y el RMSE al sacar solo 3 días de 180? ¿Por qué el RMSE cambia tanto más? ¿Cuál de las dos medidas describe mejor el error de un día normal?

_Su respuesta:_

## 7. Comprobar con scikit-learn

📘 Las mismas medidas vienen hechas en `sklearn.metrics`. Si la fórmula de `medir()` está bien, deben coincidir.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae, rmse, r2 = medir(y_pru, pred_lineal)
print("¿MAE igual?", np.isclose(mae, mean_absolute_error(y_pru, pred_lineal)))
print("¿RMSE igual?", np.isclose(rmse, mean_squared_error(y_pru, pred_lineal) ** 0.5))
print("¿R² igual?", np.isclose(r2, r2_score(y_pru, pred_lineal)))
# ✅ Debe mostrar: ¿MAE igual? True  /  ¿RMSE igual? True  /  ¿R² igual? True

### ✍️ 🧠 **De la información al conocimiento.** Lechería Los Robles debe comprometer con la planta los litros de cada día. Escriban la recomendación: ¿qué modelo usar, cómo le explicarían al dueño el error esperado en litros y qué harían con los días de falla?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema5.ipynb`, junto con el informe en PDF.